us-crude-imports-by-origin-monthly.csv was collected and cleaned into processed subfolder with data_collection.py
From now on, the rest of the csvs are cleaned and concatenated in this notebook.

Starting off with china-quantities-monthly.csv, combing all the data from Jodi data (all primaryyear csv...)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT_DIR = Path.cwd().parent
JODI_DIR = ROOT_DIR / 'data' / 'raw' / 'quantities-china' / 'jodi_data'

# loading all the jodi primary files
df_2019 = pd.read_csv(JODI_DIR / 'jodi_primaryyear_2019.csv')
df_2020 = pd.read_csv(JODI_DIR / 'jodi_primaryyear_2020.csv')
df_2021 = pd.read_csv(JODI_DIR / 'jodi_primaryyear_2021.csv')
df_2022 = pd.read_csv(JODI_DIR / 'jodi_primaryyear_2022.csv')
df_2023 = pd.read_csv(JODI_DIR / 'jodi_primaryyear_2023.csv')
df_2024 = pd.read_csv(JODI_DIR / 'jodi_primaryyear_2024.csv')
df_2025 = pd.read_csv(JODI_DIR / 'jodi_primaryyear_2025.csv')
df_2026 = pd.read_csv(JODI_DIR / 'jodi_primaryyear_2026.csv')

jodi_df = pd.concat([df_2019, df_2020, df_2021, df_2022, df_2023, df_2024, df_2025, df_2026], ignore_index=True)
jodi_df['OBS_VALUE'] = jodi_df['OBS_VALUE'].replace(['-', 'x', '..'], np.nan)
jodi_df['OBS_VALUE'] = jodi_df['OBS_VALUE'].astype('float64')
jodi_df['TIME_PERIOD'] = pd.to_datetime(jodi_df['TIME_PERIOD'])
mask = (jodi_df['REF_AREA'] == "CN") & (jodi_df['ENERGY_PRODUCT'] == "CRUDEOIL") & (jodi_df['FLOW_BREAKDOWN'].isin(["TOTIMPSB", "REFINOBS"]))
quant_china_df = jodi_df[mask]
quant_china_df.groupby('UNIT_MEASURE')['OBS_VALUE'].nunique().sort_values(ascending=False) # kbd has the most unique values
mask2 = (quant_china_df['UNIT_MEASURE'] == "KBD")
quant_china_df = quant_china_df[mask2]
quant_china_df = quant_china_df.pivot_table(
    index = ['REF_AREA', 'TIME_PERIOD', 'ENERGY_PRODUCT', 'UNIT_MEASURE', 'ASSESSMENT_CODE'],
    columns='FLOW_BREAKDOWN',
    values='OBS_VALUE', 
    aggfunc='first'
).reset_index()
quant_china_df

UNIT_MEASURE
KBD        177
KL         176
KBBL       176
KTONS      176
CONVBBL      1
Name: OBS_VALUE, dtype: int64


FLOW_BREAKDOWN,REF_AREA,TIME_PERIOD,ENERGY_PRODUCT,UNIT_MEASURE,ASSESSMENT_CODE,REFINOBS,TOTIMPSB
0,CN,2019-01-01,CRUDEOIL,KBD,1,13880.1368,10228.1652
1,CN,2019-02-01,CRUDEOIL,KBD,1,13659.9043,10066.0457
2,CN,2019-03-01,CRUDEOIL,KBD,1,12853.4477,9289.3161
3,CN,2019-04-01,CRUDEOIL,KBD,1,14654.8840,10670.1200
4,CN,2019-05-01,CRUDEOIL,KBD,1,13567.9742,9499.4710
...,...,...,...,...,...,...,...
98,CN,2026-01-01,CRUDEOIL,KBD,3,16414.5097,12010.9394
99,CN,2026-02-01,CRUDEOIL,KBD,3,16458.7586,12043.4914
100,CN,2026-03-01,CRUDEOIL,KBD,3,16257.4839,11802.2013
101,CN,2026-04-01,CRUDEOIL,KBD,3,13693.7680,9386.9240


In [ ]:
# collecting all data from gacc and merging on top of preivous dataframe comprised of data from jodi, quant_china_df
GACC_DIR = ROOT_DIR / 'data' / 'raw' / 'quantities-china' / 'gacc_table14'
import re

records = []
for path in sorted(GACC_DIR.glob('gacc-table14-import-commodities-*.xls')):
    raw = pd.read_excel(path, header=None)
    # lambda: check each row (axis=1) for a cell containing the label
    is_crude = raw.apply(
        lambda row: row.astype(str).str.contains('Crude petroleum oils', na=False).any(), 
        axis = 1
    )
    refined_petro = raw.apply(
        lambda row: row.astype(str).str.contains('Refined petroleum products', na=False).any(),
        axis = 1
    )
    cells_crude = raw[is_crude].iloc[0].dropna().tolist()
    cells_petro = raw[refined_petro].iloc[0].dropna().tolist()
    # regex to find YYYY-MM anywhere in the filename
    period = re.search(r'\d{4}-\d{2}', path.name).group(0)
    records.append({
        'period': period, 
        'crude_qty_10000t':cells_crude[2],
        'crude_value_usd_1000':cells_crude[3],
        'refined_qty_10000t': cells_petro[2],
        'refined_value_usd_1000': cells_petro[3]
    })

gacc_df = pd.DataFrame(records)
gacc_df['period'] = pd.to_datetime(gacc_df['period'])

columns = ['crude_qty_10000t', 'crude_value_usd_1000', 'refined_qty_10000t', 'refined_value_usd_1000']
for col in columns:
    gacc_df[col] = gacc_df[col].str.replace(',', '', regex=False)
    gacc_df[col] = pd.to_numeric(gacc_df[col], errors='coerce')
gacc_df

,period,crude_qty_10000t,crude_value_usd_1000,refined_qty_10000t,refined_value_usd_1000
0,2025-06-01,4989,24208656,338,1920095
1,2025-07-01,4720,23868482,403,2254841
2,2025-08-01,4949,24929306,334,1946011
3,2025-09-01,4725,23828847,395,2171769
4,2025-10-01,4836,24217445,351,1898673
5,2025-11-01,5089,24510819,424,2256576
6,2025-12-01,5597,26642861,399,1957296
7,2026-01-01,4889,22117222,426,2207562
8,2026-02-01,4805,22203513,478,2464682
9,2026-03-01,4998,27055087,364,2472628


In [ ]:
import matplotlib.pyplot as plt
OUT = ROOT_DIR / 'data' / 'processed'

# merge both gacc_df and quant_china_df
merged_df = pd.merge(quant_china_df, gacc_df, left_on='TIME_PERIOD', right_on='period', how='outer')
# density factor 7.32 bbl/tonne, qty_10000t * 10 gives thousand tonnes, * 7.32 gives thousand barrels for the month, / days_in_month gives kb/d = crude_kbd
merged_df['days'] = merged_df['TIME_PERIOD'].dt.days_in_month
merged_df['gacc_crude_kbd'] = merged_df['crude_qty_10000t'] * 10 * 7.32 / merged_df['days']
merged_df['discrepancy_crude'] = merged_df['gacc_crude_kbd'] - merged_df['TOTIMPSB']
merged_df.loc[merged_df['discrepancy_crude'].notna(), ['period', 'gacc_crude_kbd', 'TOTIMPSB', 'discrepancy_crude']]
merged_df['cny_affected'] = merged_df['TIME_PERIOD'].dt.month.isin([1,2])

merged_df['year'] = merged_df['TIME_PERIOD'].dt.year
merged_df['month'] = merged_df['TIME_PERIOD'].dt.month
merged_df['ratio'] = merged_df['TOTIMPSB'] / merged_df.groupby('year')['TOTIMPSB'].transform('mean')
seasonal = (
    merged_df[merged_df['year'] <= 2025]
    .groupby('month')['ratio']
    .mean()
)
print(seasonal.round(3))
merged_df.drop(columns='cny_affected')
merged_df.rename(columns={'period': 'PERIOD', 'TIME_PERIOD':'period'}, inplace=True)
merged_df.to_csv(OUT / 'china-quantities-monthly.csv', index=False)

month
1.0     0.967
2.0     0.999
3.0     1.017
4.0     0.980
5.0     1.001
6.0     1.022
7.0     0.951
8.0     1.018
9.0     0.999
10.0    0.973
11.0    1.041
12.0    1.034
Name: ratio, dtype: float64


In [211]:
# brent + wti merged dataset, with spread = brent - wti
PRICES_CRUDE_DIR = ROOT_DIR / 'data' / 'raw' / 'prices-crude'
OUT = ROOT_DIR / 'data' / 'processed'

brent_df = pd.read_csv(PRICES_CRUDE_DIR / 'FRED-DCOILBRENTEU-2026-08-04.csv', parse_dates=['observation_date'])
wti_df = pd.read_csv(PRICES_CRUDE_DIR / 'FRED-DCOILWTICO-2026-08-04.csv', parse_dates=['observation_date'])
print(brent_df['DCOILBRENTEU'].isna().sum(), wti_df['DCOILWTICO'].isna().sum())
fred_df = pd.merge(brent_df, wti_df, on='observation_date', how='outer')
fred_df[(fred_df['DCOILBRENTEU'].isna()) | (fred_df['DCOILWTICO'].isna())] # every null is a trading holiday
fred_df.dropna(axis=0, thresh=2, inplace=True)
fred_df['spread'] = fred_df['DCOILBRENTEU'] - fred_df['DCOILWTICO']
fred_df.isna().sum()
fred_df.to_csv(OUT / 'crude-prices-daily.csv', index=False)

56 83


In [224]:
# portwatch calls and trade volumes joined on date
TRANSIT_CALLS_DIR = ROOT_DIR / 'data' / 'raw' / 'flow-checkpoints'
OUT = ROOT_DIR / 'data' / 'processed'

# metric tonnes unit for transit volumes
calls_df = pd.read_csv(TRANSIT_CALLS_DIR / 'transit-calls-portwatch-08-02-2026.csv', parse_dates=['DateTime'])
volume_df = pd.read_csv(TRANSIT_CALLS_DIR / 'transit-trade-volume-portwatch-imf-08-02-2026.csv', parse_dates=['DateTime'])
# print(calls_df.columns, volume_df.columns)
calls_df.rename(columns={
    'Container' : 'Container_calls',
    'Dry Bulk' : 'Dry_Bulk_calls',
    'General Cargo' : 'General_Cargo_calls', 
    'Roll-on/roll-off' : 'Roll-on/roll-off_calls', 
    'Tanker' : 'Tanker_calls', 
}, inplace=True)
volume_df.rename(columns={
    'Container' : 'Container_volume',
    'Dry Bulk' : 'Dry_Bulk_volume',
    'General Cargo' : 'General_Cargo_volume', 
    'Roll-on/roll-off' : 'Roll-on/roll-off_volume', 
    'Tanker' : 'Tanker_volume', 
}, inplace=True)
calls_df.drop(columns=['7-day Moving Average', 'Prior Year: 7-day Moving Average'], inplace=True)
volume_df.drop(columns=['7-day Moving Average', 'Prior Year: 7-day Moving Average'], inplace=True)
# print(calls_df.isna().sum(), volume_df.isna().sum()) # no na values in any column
portwatch_df = pd.merge(calls_df, volume_df, on='DateTime', how='inner')
# volume per call for tankers
portwatch_df['vol_per_call_7d'] = portwatch_df['Tanker_volume'].rolling(7).sum() / portwatch_df['Tanker_calls'].rolling(7).sum()
portwatch_df.to_csv(OUT / 'hormuz-transits-daily.csv', index=False)

In [243]:
# us quantities
US_QUANT_DIR = ROOT_DIR / 'data' / 'raw' / 'quantities-us'
OUT = ROOT_DIR / 'data' / 'processed'

# weekly stocks, thousand barrels
col_finder_1 = pd.read_excel(US_QUANT_DIR / 'EIA-petroleum-balance-2026-08-02.xls', sheet_name=1, header=2,
usecols=[
    'Date',
    'Weekly U.S. Ending Stocks excluding SPR of Crude Oil  (Thousand Barrels)',
    'Weekly U.S. Ending Stocks of Crude Oil in SPR  (Thousand Barrels)',
    'Weekly U.S. Ending Stocks of Total Gasoline  (Thousand Barrels)',
    'Weekly U.S. Ending Stocks of Distillate Fuel Oil  (Thousand Barrels)',
])
# weekly flows, kb/d
col_finder_2 = pd.read_excel(US_QUANT_DIR / 'EIA-petroleum-balance-2026-08-02.xls', sheet_name=2, header=2,
usecols=[
    'Date',
    'Weekly U.S. Field Production of Crude Oil  (Thousand Barrels per Day)',
    'Weekly U.S. Imports of Crude Oil  (Thousand Barrels per Day)',
    'Weekly U.S. Exports of Crude Oil  (Thousand Barrels per Day)',
    'Weekly U.S. Refiner Net Input of Crude Oil  (Thousand Barrels per Day)',
    'Weekly U.S. Product Supplied of Petroleum Products  (Thousand Barrels per Day)',
    'Weekly U.S. Product Supplied of Finished Motor Gasoline  (Thousand Barrels per Day)',
    'Weekly U.S. Product Supplied of Distillate Fuel Oil  (Thousand Barrels per Day)',
])
# print(col_finder_1.shape, col_finder_2.shape) 2287 rows for each
merged_quant_df = pd.merge(col_finder_1, col_finder_2, on='Date', how='inner')
us_quant_df = merged_quant_df[merged_quant_df['Date'] >= '2019-01-01']
rename_map = {
    'Date': 'date',
    # Data 1 — stocks, thousand barrels
    'Weekly U.S. Ending Stocks excluding SPR of Crude Oil  (Thousand Barrels)': 'crude_stocks_ex_spr_kbbl',
    'Weekly U.S. Ending Stocks of Crude Oil in SPR  (Thousand Barrels)': 'crude_stocks_spr_kbbl',
    'Weekly U.S. Ending Stocks of Total Gasoline  (Thousand Barrels)': 'gasoline_stocks_kbbl',
    'Weekly U.S. Ending Stocks of Distillate Fuel Oil  (Thousand Barrels)': 'distillate_stocks_kbbl',
    # Data 2 — flows, thousand barrels per day
    'Weekly U.S. Field Production of Crude Oil  (Thousand Barrels per Day)': 'crude_production_kbd',
    'Weekly U.S. Imports of Crude Oil  (Thousand Barrels per Day)': 'crude_imports_kbd',
    'Weekly U.S. Exports of Crude Oil  (Thousand Barrels per Day)': 'crude_exports_kbd',
    'Weekly U.S. Refiner Net Input of Crude Oil  (Thousand Barrels per Day)': 'refinery_input_kbd',
    'Weekly U.S. Product Supplied of Petroleum Products  (Thousand Barrels per Day)': 'product_supplied_total_kbd',
    'Weekly U.S. Product Supplied of Finished Motor Gasoline  (Thousand Barrels per Day)': 'product_supplied_gasoline_kbd',
    'Weekly U.S. Product Supplied of Distillate Fuel Oil  (Thousand Barrels per Day)': 'product_supplied_distillate_kbd',
}
us_quant_df.rename(columns=rename_map, inplace=True)
us_quant_df.to_csv(OUT / 'us-quantities-weekly.csv', index=False)

In [ ]:
# us retail prices weekly 
US_RETAIL_PRICES_DIR = ROOT_DIR / 'data' / 'raw' / 'prices-retail-us'
OUT = ROOT_DIR / 'data' / 'processed'

# sheets_gasoline = pd.read_excel(US_RETAIL_PRICES_DIR / 'EIA-gasoline-regular-2026-08-02.xls', sheet_name=None, header=2)
# for name, frame in sheets_gasoline.items():
#     print(name, frame.shape)
#     print(frame.columns.tolist())
#     print()
# sheets_diesel = pd.read_excel(US_RETAIL_PRICES_DIR / 'EIA-diesel-onhighway-2026-08-02.xls', sheet_name=None, header=2)
# for name, frame in sheets_diesel.items():
#     print(name, frame.shape)
#     print(frame.columns.tolist())
#     print()
gasoline_df = pd.read_excel(US_RETAIL_PRICES_DIR / 'EIA-gasoline-regular-2026-08-02.xls', sheet_name='Data 3', header=2,
usecols=['Date', 'Weekly U.S. Regular All Formulations Retail Gasoline Prices  (Dollars per Gallon)'])
diesel_df = pd.read_excel(US_RETAIL_PRICES_DIR / 'EIA-diesel-onhighway-2026-08-02.xls', sheet_name='Data 5', header=2, 
usecols=['Date', 'Weekly U.S. No 2 Diesel Ultra Low Sulfur (0-15 ppm) Retail Prices  (Dollars per Gallon)'])


,Date,Weekly U.S. Regular All Formulations Retail Gasoline Prices (Dollars per Gallon)
0,1990-08-20,1.191
1,1990-08-27,1.245
2,1990-09-03,1.242
3,1990-09-10,1.252
4,1990-09-17,1.266
...,...,...
1871,2026-06-29,3.831
1872,2026-07-06,3.777
1873,2026-07-13,3.855
1874,2026-07-20,4.001
